# Notebook 1 — Get Sentinel-1 SAR data

**Goal:** download two radar snapshots of the Yamuna floodplain in Delhi — one *before* the July 2023 flood and one *during/after* it — so we can compare them in Notebook 2.

By the end of this notebook you will have two files in `data/raw/`:
- `s1_vv_pre.tif`  — radar backscatter, June 2023 (dry)
- `s1_vv_post.tif` — radar backscatter, mid-July 2023 (flooded)

## What is Sentinel-1 SAR, and why use it for floods?

**Sentinel-1** is a pair of European satellites carrying a **SAR** (Synthetic Aperture Radar) sensor. Instead of taking a photo with sunlight like a normal camera, it sends out its own microwave pulses and measures how much bounces back. That has two superpowers for flood mapping:

1. **It sees through clouds and works at night.** Floods happen during heavy monsoon cloud cover — exactly when optical satellites (like normal photos) are useless. Radar doesn't care about clouds.
2. **Water looks special to radar.** Calm, open water acts like a *mirror*: the radar pulse bounces *away* from the satellite instead of back. So flooded ground suddenly appears **dark** (low backscatter) compared to before. That darkening is the signal we hunt for.

**`VV` polarization** just means the pulse is sent and received as a vertical wave. It's the channel that best separates smooth water from rough land for open-water flooding.

**Why a composite (a median of several passes) instead of one image?** A single radar image is noisy (see 'speckle' in Notebook 2). Taking the *median* of all passes in a date window calms that noise down and fills small gaps.

**What do the pixel numbers mean?** Sentinel-1 backscatter is stored in **decibels (dB)** — a logarithmic scale. The values are **negative** (roughly -25 to 0). That is completely normal: more negative = darker = smoother surface (like water).

## Step 1 — Imports and folders

We import Earth Engine (`ee`) plus `geemap` (interactive maps + easy downloads), and figure out where the project's `data/` folder is so our files land in the right place.

In [1]:
import ee
import geemap
import geopandas as gpd
from shapely.geometry import box
from pathlib import Path

# Find the repo root whether you launched Jupyter from the repo folder or from notebooks/
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = REPO / "data"
RAW = DATA / "raw"
RAW.mkdir(parents=True, exist_ok=True)

print("Repo root: ", REPO)
print("Raw data:  ", RAW)

Repo root:  /Users/user/yamuna-flood-mapper
Raw data:   /Users/user/yamuna-flood-mapper/data/raw


## Step 2 — Connect to Google Earth Engine

Earth Engine is Google's cloud archive of satellite imagery. We don't download the whole archive — we ask Google's servers to do the filtering and only send us the small result.

This cell **initializes** the connection. Your machine is already authenticated, so it connects straight away. (On a fresh machine it would fall back to `ee.Authenticate()`, a one-time browser sign-in.) `ee.Initialize(project=...)` connects using your approved Earth Engine Cloud project.

In [2]:
EE_PROJECT = "urban-flood-analysis-ncr-in"   # your Earth Engine Cloud project

# Connect to Earth Engine. We try to initialize first; only if that fails
# (e.g. on a brand-new machine) do we run the one-time browser sign-in.
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)

print("Earth Engine connected for project:", EE_PROJECT)

Earth Engine connected for project: urban-flood-analysis-ncr-in


## Step 3 — Define the Area of Interest (AOI)

Our AOI is a tight box on the **Yamuna river corridor through Delhi** — the stretch hit hardest in July 2023 (Wazirabad down past Okhla, including the Mayur Vihar east bank). Keeping it tight keeps files small and processing fast.

We also save the box to `data/aoi.geojson` so the whole project is reproducible — anyone can see exactly what area we mapped.

In [3]:
# (lon_min, lat_min, lon_max, lat_max)
LON_MIN, LAT_MIN, LON_MAX, LAT_MAX = 77.18, 28.50, 77.38, 28.78

aoi = ee.Geometry.Rectangle([LON_MIN, LAT_MIN, LON_MAX, LAT_MAX])

# Save a reproducible copy as GeoJSON
aoi_gdf = gpd.GeoDataFrame(
    {"name": ["yamuna_delhi_aoi"]},
    geometry=[box(LON_MIN, LAT_MIN, LON_MAX, LAT_MAX)],
    crs="EPSG:4326",
)
aoi_path = DATA / "aoi.geojson"
aoi_gdf.to_file(aoi_path, driver="GeoJSON")
print("Saved AOI to:", aoi_path)

Saved AOI to: /Users/user/yamuna-flood-mapper/data/aoi.geojson


## Step 4 — Build the pre-flood and post-flood radar composites

We pull the `COPERNICUS/S1_GRD` collection and filter it down to what we need:
- only images **covering our AOI**
- only the **IW** mode and **VV** polarization (the standard land-mapping setup)
- only **one orbit direction** (ascending *or* descending). This matters: radar looks sideways, so images from opposite passes aren't directly comparable. Keeping one direction makes the before/after fair.

Then we take the **median** over each date window to get one clean composite per period.

- **Pre-flood:** 15 May – 30 June 2023 (dry, before the monsoon peak)
- **Post-flood:** 11 – 24 July 2023 (the Yamuna peaked ~13 July at a record 208.66 m)

In [4]:
def s1_composite(start, end, aoi, orbit):
    """Return (collection, median VV composite in dB) for a date window."""
    col = (
        ee.ImageCollection("COPERNICUS/S1_GRD")
        .filterBounds(aoi)
        .filterDate(start, end)
        .filter(ee.Filter.eq("instrumentMode", "IW"))
        .filter(ee.Filter.listContains("transmitterReceiverPolarisation", "VV"))
        .filter(ee.Filter.eq("orbitProperties_pass", orbit))
        .select("VV")
    )
    return col, col.median().clip(aoi)


ORBIT = "DESCENDING"   # if counts below are 0, change to "ASCENDING" and re-run

pre_col,  pre_vv  = s1_composite("2023-05-15", "2023-06-30", aoi, ORBIT)
post_col, post_vv = s1_composite("2023-07-11", "2023-07-24", aoi, ORBIT)

print("Pre-flood  images found:", pre_col.size().getInfo())
print("Post-flood images found:", post_col.size().getInfo())

Pre-flood  images found: 7


Post-flood images found: 2


> **Troubleshooting:** if either count is `0`, this orbit direction has no coverage for that window. Run the cell below to see which directions *are* available, then set `ORBIT` accordingly and re-run Step 4.

In [5]:
probe = (
    ee.ImageCollection("COPERNICUS/S1_GRD")
    .filterBounds(aoi)
    .filterDate("2023-05-15", "2023-07-24")
    .filter(ee.Filter.eq("instrumentMode", "IW"))
)
print("Orbit directions available:",
      probe.aggregate_array("orbitProperties_pass").distinct().getInfo())

Orbit directions available: ['ASCENDING', 'DESCENDING']


## Step 5 — Preview the two composites side by side

Let's eyeball the before/after. We pull a small thumbnail of each composite straight from Earth Engine and show them with `matplotlib`.

We use a **static image** (not an interactive map) on purpose: static plots render for anyone viewing this notebook on GitHub, whereas interactive map widgets show up blank there.

Look at the river and the low-lying east bank: in the **post-flood** image those areas should look **darker** (water = low backscatter) than in the **pre-flood** image. Darker = our flood signal.

In [6]:
import io
import urllib.request
import matplotlib.pyplot as plt
from PIL import Image


def ee_thumbnail(img, aoi):
    """Fetch a small rendered PNG thumbnail of an EE image as a PIL image."""
    url = img.getThumbURL({"min": -25, "max": 0, "dimensions": 600, "region": aoi})
    return Image.open(io.BytesIO(urllib.request.urlopen(url).read()))


fig, axes = plt.subplots(1, 2, figsize=(12, 7))
for ax, img, title in zip(
    axes,
    [pre_vv, post_vv],
    ["Pre-flood VV (Jun 2023)", "Post-flood VV (Jul 2023)"],
):
    ax.imshow(ee_thumbnail(img, aoi))
    ax.set_title(title)
    ax.axis("off")

fig.suptitle("Sentinel-1 VV backscatter — darker = smoother surface (water)", y=0.98)
plt.tight_layout()
plt.show()

/var/folders/jr/0tppzf_n6rscbz7qzblfgwbc0000gn/T/ipykernel_14655/3509343356.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 6 — Download both composites to `data/raw/`

Now we pull the two composites down as **GeoTIFFs** (a GeoTIFF is an image where every pixel also knows its real-world location). `geemap.download_ee_image` fetches them straight to your laptop at 10 m resolution — no Google Drive round-trip needed.

> This takes a minute or two each and prints a progress bar. If you ever get a size/timeout error, it's safe to just re-run this cell.

In [7]:
pre_path  = RAW / "s1_vv_pre.tif"
post_path = RAW / "s1_vv_post.tif"

geemap.download_ee_image(pre_vv,  str(pre_path),  scale=10, region=aoi, crs="EPSG:4326")
geemap.download_ee_image(post_vv, str(post_path), scale=10, region=aoi, crs="EPSG:4326")

print("Saved:")
print("  ", pre_path,  "exists:", pre_path.exists())
print("  ", post_path, "exists:", post_path.exists())

/Users/user/miniconda3/envs/yamuna-flood/lib/python3.11/site-packages/geemap/common.py:12434: FutureWarning: 'BaseImage' is deprecated and will be removed in a future release.  Please use the 'ee.Image.gd' accessor instead.
  img = gd.download.BaseImage(image)


  0%|          |0/21 tiles [00:00<?]

/Users/user/miniconda3/envs/yamuna-flood/lib/python3.11/site-packages/geedim/image.py:254: RuntimeWarning: Couldn't find STAC entry for: 'None'.
  return STACClient().get(self.id)


  0%|          |0/21 tiles [00:00<?]

Saved:
   /Users/user/yamuna-flood-mapper/data/raw/s1_vv_pre.tif exists: True
   /Users/user/yamuna-flood-mapper/data/raw/s1_vv_post.tif exists: True


## Recap

You just: connected to Earth Engine, defined the Delhi Yamuna AOI, built a clean *before* and *after* radar composite filtered to one orbit/polarization, previewed them, and downloaded both as GeoTIFFs.

**Next — Notebook 2:** load these two files, clean the radar speckle, subtract them, and turn the darkening signal into a binary **flood mask**.